# Alibaba 2023 GPU Trace EDA

This notebook explores the Alibaba `cluster-trace-gpu-v2023` workload traces as a candidate source for realistic job and cluster assumptions. The goal is not to change the MILP formulation yet. The goal is to understand whether trace-derived groups can support a credible business/workload definition for later experiments.

Key questions:

- What does the GPU cluster inventory look like by GPU type?
- How many jobs have no GPU-type constraint versus explicit GPU specs?
- Can jobs be grouped into 4 resource/workload proxy classes using GPU constraints, GPU demand, duration, CPU, memory, QoS, and phase?
- Which trace fields can map cleanly to the current MILP schema, and which ones still require assumptions?

Source: https://github.com/alibaba/clusterdata/tree/master/cluster-trace-gpu-v2023

In [ ]:
from __future__ import annotations

from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "alibaba_gpu_v2023"
RAW_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)
plt.style.use("default")

## Load Trace Files

The notebook uses local CSVs under `data/raw/alibaba_gpu_v2023` when present. If they are missing, it downloads the four small 2023 CSVs directly from GitHub raw URLs.

Files used:

- `openb_node_list_gpu_node.csv`: GPU-node inventory.
- `openb_node_list_all_node.csv`: full node inventory, including non-GPU nodes.
- `openb_pod_list_default.csv`: default submitted pod/task list.
- `openb_pod_list_gpuspec33.csv`: pod/task list augmented with GPU-type constraints for about one third of GPU tasks.

In [ ]:
BASE_URL = "https://raw.githubusercontent.com/alibaba/clusterdata/master/cluster-trace-gpu-v2023/csv"
TRACE_FILES = {
    "nodes_gpu": "openb_node_list_gpu_node.csv",
    "nodes_all": "openb_node_list_all_node.csv",
    "pods_default": "openb_pod_list_default.csv",
    "pods_gpuspec": "openb_pod_list_gpuspec33.csv",
}


def ensure_trace_file(filename: str) -> Path:
    path = RAW_DIR / filename
    if not path.exists():
        url = f"{BASE_URL}/{filename}"
        print(f"Downloading {url}")
        urlretrieve(url, path)
    return path


paths = {name: ensure_trace_file(filename) for name, filename in TRACE_FILES.items()}
paths

In [ ]:
nodes_gpu = pd.read_csv(paths["nodes_gpu"])
nodes_all = pd.read_csv(paths["nodes_all"])
pods_default = pd.read_csv(paths["pods_default"])
pods_gpuspec = pd.read_csv(paths["pods_gpuspec"])

print("nodes_gpu", nodes_gpu.shape)
print("nodes_all", nodes_all.shape)
print("pods_default", pods_default.shape)
print("pods_gpuspec", pods_gpuspec.shape)

display(nodes_gpu.head())
display(pods_gpuspec.head())

## Basic Cleaning

The trace uses seconds for timestamps. We derive runtime and wait-time features. We also distinguish jobs with no GPU-type constraint from jobs that require one or more specific GPU types.

In [ ]:
def normalize_gpu_spec(value: object) -> str:
    if pd.isna(value):
        return "NO_GPU_TYPE_CONSTRAINT"
    text = str(value).strip()
    if text == "" or text.lower() == "nan":
        return "NO_GPU_TYPE_CONSTRAINT"
    return text


def add_pod_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["gpu_spec_clean"] = out["gpu_spec"].map(normalize_gpu_spec)
    out["has_gpu_type_constraint"] = out["gpu_spec_clean"] != "NO_GPU_TYPE_CONSTRAINT"
    out["cpu_cores"] = out["cpu_milli"] / 1000.0
    out["memory_gib"] = out["memory_mib"] / 1024.0
    out["effective_gpu"] = out["num_gpu"] * out["gpu_milli"] / 1000.0
    out["runtime_seconds"] = out["deletion_time"] - out["scheduled_time"]
    out["wall_time_seconds"] = out["deletion_time"] - out["creation_time"]
    out["wait_seconds"] = out["scheduled_time"] - out["creation_time"]
    for column in ["runtime_seconds", "wall_time_seconds", "wait_seconds"]:
        out.loc[out[column] < 0, column] = np.nan
    out["runtime_hours"] = out["runtime_seconds"] / 3600.0
    out["wall_time_hours"] = out["wall_time_seconds"] / 3600.0
    out["wait_hours"] = out["wait_seconds"] / 3600.0
    out["duration_h_for_milp"] = np.ceil(out["runtime_hours"].clip(lower=1 / 3600)).astype("Int64")
    return out


pods = add_pod_features(pods_gpuspec)
gpu_pods = pods[pods["num_gpu"] > 0].copy()

print("All pods:", len(pods))
print("GPU pods:", len(gpu_pods))
print("GPU pods with GPU-type constraint:", int(gpu_pods["has_gpu_type_constraint"].sum()))
print("GPU pods without GPU-type constraint:", int((~gpu_pods["has_gpu_type_constraint"]).sum()))

## Cluster Inventory

This summarizes physical GPU-node availability by model. Alibaba `G1`, `G2`, and `G3` are undisclosed internal GPU codes, so they should be treated as resource classes unless we find a defensible mapping elsewhere.

In [ ]:
node_inventory = (
    nodes_gpu.groupby("model", dropna=False)
    .agg(nodes=("sn", "count"), total_gpus=("gpu", "sum"), median_gpus_per_node=("gpu", "median"))
    .sort_values("total_gpus", ascending=False)
)
display(node_inventory)

ax = node_inventory["total_gpus"].plot(kind="bar", figsize=(8, 4), title="GPU Inventory by Node Model")
ax.set_xlabel("GPU model / resource class")
ax.set_ylabel("Total GPUs")
plt.tight_layout()

## Job GPU-Spec Distribution

This table is central for deciding whether GPU spec can support cluster compatibility and workload proxy labels. The `NO_GPU_TYPE_CONSTRAINT` group is not necessarily preprocessing; it means the scheduler did not need a specific GPU type.

In [ ]:
gpu_spec_summary = (
    gpu_pods.groupby("gpu_spec_clean")
    .agg(
        jobs=("name", "count"),
        median_num_gpu=("num_gpu", "median"),
        mean_effective_gpu=("effective_gpu", "mean"),
        median_cpu_cores=("cpu_cores", "median"),
        median_memory_gib=("memory_gib", "median"),
        median_runtime_hours=("runtime_hours", "median"),
        median_wait_hours=("wait_hours", "median"),
    )
    .sort_values("jobs", ascending=False)
)
display(gpu_spec_summary)

ax = gpu_spec_summary.head(20)["jobs"].plot(kind="bar", figsize=(12, 4), title="Top GPU-Spec Constraints by Job Count")
ax.set_xlabel("GPU spec")
ax.set_ylabel("Jobs")
plt.xticks(rotation=70, ha="right")
plt.tight_layout()

## Numeric Feature Distributions

These distributions are candidates for calibrating the synthetic generator later. They are also candidates for clustering features.

In [ ]:
numeric_columns = ["num_gpu", "gpu_milli", "effective_gpu", "cpu_cores", "memory_gib", "runtime_hours", "wait_hours"]
display(gpu_pods[numeric_columns].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).T)

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
plot_columns = ["num_gpu", "effective_gpu", "cpu_cores", "memory_gib", "runtime_hours", "wait_hours"]
for ax, column in zip(axes.ravel(), plot_columns):
    series = gpu_pods[column].replace([np.inf, -np.inf], np.nan).dropna()
    if column in {"runtime_hours", "wait_hours", "cpu_cores", "memory_gib"}:
        series = np.log1p(series)
        ax.set_xlabel(f"log1p({column})")
    else:
        ax.set_xlabel(column)
    ax.hist(series, bins=40)
    ax.set_title(column)
plt.tight_layout()

## Constraint-First 4-Group Proposal

This is the first pass we can explain in a thesis/business discussion. It follows the hypothesis that jobs without GPU-type constraints form one broad flexible group, while constrained jobs can be split into three more specific resource/workload proxy groups.

Proposed groups:

1. `unconstrained_flexible_batch`: no GPU-type constraint. This may be preprocessing-like, generic batch, or placement-flexible ML work.
2. `constrained_low_or_shared_gpu`: constrained jobs with fractional/shared GPU demand, latency-sensitive QoS, or lower-power GPU specs such as T4/P100.
3. `constrained_medium_gpu`: constrained jobs with moderate GPU demand and no strong high-performance signal.
4. `constrained_high_perf_or_training`: constrained jobs with high GPU demand, long runtime, or high-performance GPU specs.

These labels should be treated as proxies. Later, if business validation contradicts them, we can rename them to neutral resource classes.

In [ ]:
HIGH_PERF_TOKENS = {"A10", "G3", "V100M16", "V100M32"}
LOW_OR_INFERENCE_TOKENS = {"T4", "P100"}


def spec_tokens(spec: str) -> set[str]:
    if spec == "NO_GPU_TYPE_CONSTRAINT":
        return set()
    return {item.strip() for item in spec.split("|") if item.strip()}


runtime_75 = gpu_pods["runtime_hours"].quantile(0.75)
runtime_90 = gpu_pods["runtime_hours"].quantile(0.90)


def rule_based_workload_proxy(row: pd.Series) -> str:
    tokens = spec_tokens(row["gpu_spec_clean"])
    no_constraint = row["gpu_spec_clean"] == "NO_GPU_TYPE_CONSTRAINT"
    shared_gpu = row["num_gpu"] == 1 and row["gpu_milli"] < 1000
    high_perf_spec = bool(tokens & HIGH_PERF_TOKENS)
    low_or_inference_spec = bool(tokens and tokens <= LOW_OR_INFERENCE_TOKENS)

    if no_constraint:
        return "unconstrained_flexible_batch"
    if row["num_gpu"] >= 4 or row["effective_gpu"] >= 2 or row["runtime_hours"] >= runtime_90 or (high_perf_spec and row["runtime_hours"] >= runtime_75):
        return "constrained_high_perf_or_training"
    if shared_gpu or row["qos"] == "LS" or low_or_inference_spec:
        return "constrained_low_or_shared_gpu"
    return "constrained_medium_gpu"


gpu_pods["rule_workload_proxy"] = gpu_pods.apply(rule_based_workload_proxy, axis=1)

rule_summary = (
    gpu_pods.groupby("rule_workload_proxy")
    .agg(
        jobs=("name", "count"),
        share=("name", lambda x: len(x) / len(gpu_pods)),
        median_num_gpu=("num_gpu", "median"),
        median_effective_gpu=("effective_gpu", "median"),
        median_cpu_cores=("cpu_cores", "median"),
        median_memory_gib=("memory_gib", "median"),
        median_runtime_hours=("runtime_hours", "median"),
        median_wait_hours=("wait_hours", "median"),
        gpu_type_constraint_rate=("has_gpu_type_constraint", "mean"),
    )
    .sort_values("jobs", ascending=False)
)
display(rule_summary)

display(pd.crosstab(gpu_pods["rule_workload_proxy"], gpu_pods["qos"], normalize="index"))
display(pd.crosstab(gpu_pods["rule_workload_proxy"], gpu_pods["pod_phase"], normalize="index"))

## Unsupervised Mixed-Feature Clustering

This section tries to discover 4 groups from numerical and categorical trace fields. It uses scikit-learn if available. If scikit-learn is not installed, skip this section and use the rule-based grouping above.

Features used:

- Numeric: GPU count, GPU milli, effective GPU, CPU, memory, runtime, wait time.
- Categorical: GPU spec, QoS, pod phase, GPU-type constraint flag.

Interpretation matters: clusters are statistical groups, not automatically business workload classes.

In [ ]:
try:
    from sklearn.cluster import KMeans
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import silhouette_score
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler

    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False
    print("scikit-learn is not installed. Rule-based grouping is still available.")

In [ ]:
if SKLEARN_AVAILABLE:
    cluster_df = gpu_pods.copy()
    for column in ["runtime_hours", "wait_hours", "cpu_cores", "memory_gib"]:
        cluster_df[f"log1p_{column}"] = np.log1p(cluster_df[column].clip(lower=0))

    numeric_features = [
        "num_gpu",
        "gpu_milli",
        "effective_gpu",
        "log1p_cpu_cores",
        "log1p_memory_gib",
        "log1p_runtime_hours",
        "log1p_wait_hours",
    ]
    categorical_features = ["gpu_spec_clean", "qos", "pod_phase", "has_gpu_type_constraint"]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ]
    )
    X = preprocessor.fit_transform(cluster_df[numeric_features + categorical_features])

    scores = {}
    for k in range(2, 8):
        model = KMeans(n_clusters=k, random_state=42, n_init=20)
        labels = model.fit_predict(X)
        sample_size = min(3000, X.shape[0])
        scores[k] = silhouette_score(X, labels, sample_size=sample_size, random_state=42)
    display(pd.Series(scores, name="silhouette_score"))

    kmeans = KMeans(n_clusters=4, random_state=42, n_init=30)
    cluster_df["kmeans_group"] = kmeans.fit_predict(X)
    gpu_pods["kmeans_group"] = cluster_df["kmeans_group"].values

    cluster_summary = (
        cluster_df.groupby("kmeans_group")
        .agg(
            jobs=("name", "count"),
            share=("name", lambda x: len(x) / len(cluster_df)),
            median_num_gpu=("num_gpu", "median"),
            median_gpu_milli=("gpu_milli", "median"),
            median_effective_gpu=("effective_gpu", "median"),
            median_cpu_cores=("cpu_cores", "median"),
            median_memory_gib=("memory_gib", "median"),
            median_runtime_hours=("runtime_hours", "median"),
            median_wait_hours=("wait_hours", "median"),
            gpu_type_constraint_rate=("has_gpu_type_constraint", "mean"),
        )
        .sort_values("jobs", ascending=False)
    )
    display(cluster_summary)
    display(pd.crosstab(cluster_df["kmeans_group"], cluster_df["gpu_spec_clean"]))
    display(pd.crosstab(cluster_df["kmeans_group"], cluster_df["rule_workload_proxy"], normalize="index"))

## Additional Unsupervised Clustering Alternatives

KMeans is only one view of the data. This section compares several alternatives on the same mixed-feature representation:

- `GaussianMixture`: soft probabilistic groups; useful if clusters overlap.
- `AgglomerativeClustering`: hierarchical grouping; useful for inspecting whether 4 groups are natural or forced.
- `DBSCAN`: density-based grouping; useful for detecting outliers and dense cores without forcing every job into one of four groups.
- `TruncatedSVD + KMeans`: reduces sparse one-hot categorical dimensions before clustering.

The point is not to declare one winner automatically. The point is to see whether different methods tell a consistent story about unconstrained jobs, low/shared GPU jobs, medium constrained jobs, and high-performance/long-running jobs.

In [ ]:
if SKLEARN_AVAILABLE:
    from sklearn.cluster import AgglomerativeClustering, DBSCAN
    from sklearn.decomposition import TruncatedSVD
    from sklearn.mixture import GaussianMixture
    from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score


    def dense_sample(X, max_rows: int = 2500, seed: int = 42):
        rng = np.random.default_rng(seed)
        row_count = X.shape[0]
        if row_count > max_rows:
            sample_index = np.sort(rng.choice(row_count, size=max_rows, replace=False))
            return X[sample_index].toarray() if hasattr(X, "toarray") else np.asarray(X[sample_index]), sample_index
        return X.toarray() if hasattr(X, "toarray") else np.asarray(X), np.arange(row_count)


    def evaluate_labels(X_eval, labels, method_name: str) -> dict[str, object]:
        labels = np.asarray(labels)
        non_noise_mask = labels != -1
        unique_non_noise = sorted(set(labels[non_noise_mask]))
        result = {
            "method": method_name,
            "clusters_excluding_noise": len(unique_non_noise),
            "noise_share": float((labels == -1).mean()),
            "largest_cluster_share": float(pd.Series(labels).value_counts(normalize=True).iloc[0]),
            "silhouette": np.nan,
            "calinski_harabasz": np.nan,
            "davies_bouldin": np.nan,
        }
        if len(unique_non_noise) >= 2 and non_noise_mask.sum() > len(unique_non_noise):
            X_non_noise = X_eval[non_noise_mask]
            labels_non_noise = labels[non_noise_mask]
            result["silhouette"] = silhouette_score(X_non_noise, labels_non_noise)
            result["calinski_harabasz"] = calinski_harabasz_score(X_non_noise, labels_non_noise)
            result["davies_bouldin"] = davies_bouldin_score(X_non_noise, labels_non_noise)
        return result


    X_dense, sample_index = dense_sample(X, max_rows=2500)
    comparison_rows = []

    gmm = GaussianMixture(n_components=4, covariance_type="diag", random_state=42, n_init=10)
    gmm_labels = gmm.fit_predict(X_dense)
    comparison_rows.append(evaluate_labels(X_dense, gmm_labels, "GaussianMixture_diag_k4"))

    agglomerative = AgglomerativeClustering(n_clusters=4, linkage="ward")
    agglomerative_labels = agglomerative.fit_predict(X_dense)
    comparison_rows.append(evaluate_labels(X_dense, agglomerative_labels, "Agglomerative_ward_k4"))

    dbscan_grid = []
    for eps in [1.5, 2.0, 2.5, 3.0, 3.5]:
        dbscan = DBSCAN(eps=eps, min_samples=20)
        labels = dbscan.fit_predict(X_dense)
        row = evaluate_labels(X_dense, labels, f"DBSCAN_eps_{eps}")
        dbscan_grid.append(row)
    comparison_rows.extend(dbscan_grid)

    svd_components = min(20, X.shape[1] - 1)
    svd = TruncatedSVD(n_components=svd_components, random_state=42)
    X_svd = svd.fit_transform(X)
    X_svd_dense, svd_sample_index = dense_sample(X_svd, max_rows=2500)
    svd_kmeans = KMeans(n_clusters=4, random_state=42, n_init=30)
    svd_kmeans_labels = svd_kmeans.fit_predict(X_svd_dense)
    comparison_rows.append(evaluate_labels(X_svd_dense, svd_kmeans_labels, "TruncatedSVD20_KMeans_k4"))

    clustering_comparison = pd.DataFrame(comparison_rows).sort_values("silhouette", ascending=False, na_position="last")
    display(clustering_comparison)

    sampled_jobs = cluster_df.iloc[sample_index].copy()
    sampled_jobs["gmm_group"] = gmm_labels
    sampled_jobs["agglomerative_group"] = agglomerative_labels
    sampled_jobs["svd_kmeans_group"] = svd_kmeans_labels[: len(sampled_jobs)]

    for label_column in ["gmm_group", "agglomerative_group", "svd_kmeans_group"]:
        print(f"\n{label_column} summary")
        display(
            sampled_jobs.groupby(label_column)
            .agg(
                jobs=("name", "count"),
                median_num_gpu=("num_gpu", "median"),
                median_effective_gpu=("effective_gpu", "median"),
                median_cpu_cores=("cpu_cores", "median"),
                median_memory_gib=("memory_gib", "median"),
                median_runtime_hours=("runtime_hours", "median"),
                gpu_type_constraint_rate=("has_gpu_type_constraint", "mean"),
            )
            .sort_values("jobs", ascending=False)
        )
        display(pd.crosstab(sampled_jobs[label_column], sampled_jobs["rule_workload_proxy"], normalize="index"))

    EXPORT_DIR = PROJECT_ROOT / "experiments" / "outputs" / "alibaba_2023_eda"
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    clustering_comparison.to_csv(EXPORT_DIR / "clustering_method_comparison.csv", index=False)
else:
    print("Install scikit-learn to run the additional clustering alternatives.")

## Mapping Alternatives for Our Problem

The trace is naturally GPU-scheduling data, not energy-scheduling data. A defensible integration should keep these mappings separate.

### Alternative A: Cluster-Level Energy Assignment

- Aggregate Alibaba nodes into thesis clusters by GPU resource class.
- Use job GPU-type constraints to define compatibility.
- Use trace-derived duration, GPU demand, CPU, memory, and arrival patterns.
- Add energy assumptions externally: kW/GPU by type, server overhead, PUE.

This is closest to the current MILP.

### Alternative B: Two-Stage Scheduling

1. Assign jobs to energy-aware clusters.
2. Schedule jobs within each cluster using lower-level GPU placement.

This may be more realistic, but it is more complex and should not be the first implementation.

### Alternative C: Trace-Calibrated Synthetic Generator

- Use Alibaba distributions for duration, GPU demand, compatibility, and arrivals.
- Keep generator feasible-by-construction.
- Use this for controlled Gurobi stress tests while remaining realistic.

This is likely the best bridge between research realism and solver-limit experiments.

In [ ]:
candidate_mapping_table = pd.DataFrame(
    [
        {
            "trace_signal": "num_gpu, gpu_milli",
            "model_use": "GPU demand / capacity constraint",
            "confidence": "high",
        },
        {
            "trace_signal": "scheduled_time, deletion_time",
            "model_use": "job duration distribution",
            "confidence": "high if phases are filtered carefully",
        },
        {
            "trace_signal": "creation_time",
            "model_use": "arrival-time distribution",
            "confidence": "medium-high",
        },
        {
            "trace_signal": "gpu_spec",
            "model_use": "cluster compatibility / resource class",
            "confidence": "high for compatibility, low for business workload labels",
        },
        {
            "trace_signal": "qos",
            "model_use": "priority or latency-sensitivity proxy",
            "confidence": "medium",
        },
        {
            "trace_signal": "GPU model",
            "model_use": "cluster resource class and possible power assumption",
            "confidence": "medium because G1/G2/G3 are undisclosed",
        },
        {
            "trace_signal": "none",
            "model_use": "job power / energy consumption",
            "confidence": "requires external power model",
        },
    ]
)
display(candidate_mapping_table)

## Export Optional Grouped Job Table

This exports a lightweight grouped table for later discussion. It is not yet a MILP-ready input because power assumptions and final business categories are still open.

In [ ]:
EXPORT_DIR = PROJECT_ROOT / "experiments" / "outputs" / "alibaba_2023_eda"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

export_columns = [
    "name",
    "cpu_milli",
    "memory_mib",
    "num_gpu",
    "gpu_milli",
    "gpu_spec_clean",
    "qos",
    "pod_phase",
    "creation_time",
    "scheduled_time",
    "deletion_time",
    "runtime_hours",
    "wait_hours",
    "rule_workload_proxy",
]
if "kmeans_group" in gpu_pods.columns:
    export_columns.append("kmeans_group")

gpu_pods[export_columns].to_csv(EXPORT_DIR / "gpu_jobs_grouped_exploration.csv", index=False)
node_inventory.to_csv(EXPORT_DIR / "node_inventory_by_model.csv")
gpu_spec_summary.to_csv(EXPORT_DIR / "gpu_spec_summary.csv")
rule_summary.to_csv(EXPORT_DIR / "rule_group_summary.csv")

print(f"Wrote EDA outputs to {EXPORT_DIR}")